### `SqliteSaver`
- LangGraph 상태를  SQLite DB에 저장하는 체크포인터

<br>

- 설치

In [1]:
# ! pip install langgraph-checkpoint-sqlite

<br>

#### 사용 방식

<table>
<thead>
<tr>
<th style="text-align: left;">특징</th>
<th style="text-align: left;">방식 1 (직접 연결)</th>
<th style="text-align: left;">방식 2 (Context Manager)</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><strong>리소스 관리</strong></td>
<td style="text-align: left;">수동 (close 필요)</td>
<td style="text-align: left;">자동 (with 블록)</td>
</tr>
<tr>
<td style="text-align: left;"><strong>안전성</strong></td>
<td style="text-align: left;">낮음</td>
<td style="text-align: left;">높음 ✅</td>
</tr>
<tr>
<td style="text-align: left;"><strong>권장 상황</strong></td>
<td style="text-align: left;">연결 재사용 시</td>
<td style="text-align: left;">일반적인 사용</td>
</tr>
</tbody>
</table>

<br>

1. **직접 연결 관리**
- 데이터베이스 연결을 직접 생성하고 관리하는 방식
- 연결을 재사용하는 경우 유용
  
<br>

- 반드시 `conn.close()`를 호출하여 연결을 종료
- `try-finally` 블록으로 예외 발생 시에도 안전하게 종료되도록 구현
- 연결을 닫지 않으면 파일 잠금 문제가 발생 가능

In [2]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.agents import create_agent

- 데이터베이스 연결 생성

In [3]:
conn = sqlite3.connect("./DB/checkpoints.db", check_same_thread=False)

- **체크포인터 생성 : 연결 객체를 직접 전달하여 체크포인터 생성**

In [4]:
checkpointer = SqliteSaver(conn)

- 그래프 생성 및 컴파일

In [5]:
def chatbot(state: MessagesState):
    """간단한 챗봇 노드"""
    agent = create_agent(model="openai:gpt-4.1-mini", tools=[])
    response = agent.invoke(state)
    return {"messages": response["messages"]}

In [6]:
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

- 체크포인터 연결

In [7]:
graph = builder.compile(checkpointer=checkpointer)

- 그래프 실행

In [ ]:
try:
    config = {"configurable": {"thread_id": "user-001"}}

    # 첫 번째 메시지
    response = graph.invoke(
        {"messages": [{"role": "user", "content": "안녕하세요. 저는 김철수입니다."}]},
        config=config
    )
    print(response["messages"][-1].content)

    # 두 번째 메시지 (히스토리 유지됨)
    response = graph.invoke(
        {"messages": [{"role": "user", "content": "제 이름은 무엇인가요?"}]},
        config=config
    )
    print(response["messages"][-1].content)

finally:
    conn.close()

안녕하세요, 김철수님! 만나서 반갑습니다. 어떻게 도와드릴까요?
고객님께서 말씀해 주신 이름은 김철수입니다.


<br>

2. **Context Manager (권장)**

In [9]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.agents import create_agent

- 그래프 정의

In [10]:
def chatbot(state: MessagesState):
    """간단한 챗봇 노드"""
    agent = create_agent(model="openai:gpt-4.1-mini", tools=[])
    response = agent.invoke(state)
    return {"messages": response["messages"]}

In [11]:
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

- **Context Manager로 체크포인터 생성 및 사용**

In [13]:
with SqliteSaver.from_conn_string("./DB/checkpoints.db") as checkpointer:
    # 그래프 컴파일
    graph = builder.compile(checkpointer=checkpointer)

    # 여러 세션 처리
    # 사용자 1의 세션
    config_user1 = {"configurable": {"thread_id": "user-001"}}
    response = graph.invoke(
        {"messages": [{"role": "user", "content": "제 이름은 철수입니다"}]},
        config=config_user1
    )
    print(f"User 1: {response['messages'][-1].content}")

    # 사용자 2의 세션 (독립적인 히스토리)
    config_user2 = {"configurable": {"thread_id": "user-002"}}
    response = graph.invoke(
        {"messages": [{"role": "user", "content": "제 이름은 영희입니다"}]},
        config=config_user2
    )
    print(f"User 2: {response['messages'][-1].content}")

    # 상태 조회
    # 사용자 1의 현재 상태 확인
    state = graph.get_state(config_user1)
    print(f"User 1 메시지 수: {len(state.values['messages'])}")

User 1: 네, 알겠습니다. 앞으로는 '철수'님으로 불러드릴게요. 도움이 필요하시면 언제든 말씀해 주세요!
User 2: 안녕하세요, 영희님! 만나서 반가워요. 어떻게 도와드릴까요?
User 1 메시지 수: 6


<br>

#### 연결 문자열 형식
- `SqliteSaver`는 다양한 형식의 연결 문자열을 지원

<br>

- 파일 경로 (현재 디렉토리)

In [ ]:
with SqliteSaver.from_conn_string("./DB/checkpoints.db") as checkpointer:
    # 현재 디렉토리에 checkpoints.db 파일 생성
    pass

- 하위 폴더 경로

In [ ]:
import os

In [ ]:
os.makedirs("DB", exist_ok=True)

with SqliteSaver.from_conn_string("DB/checkpoints.db") as checkpointer:
    # DB 폴더에 checkpoints.db 파일 생성
    pass

- 인메모리 데이터베이스 (휘발성)

In [ ]:
with SqliteSaver.from_conn_string(":memory:") as checkpointer:
    # 메모리에만 저장, 프로세스 종료 시 데이터 소실
    # 테스트나 임시 작업에 유용
    pass

<br>

#### 비동기 체크포인터: `AsyncSqliteSaver`
- **동시 다발적인 요청을 처리해야 하는 웹 서버 환경에서는 비동기 버전을 사용**


<table>
<thead>
<tr>
<th style="text-align: left;">상황</th>
<th style="text-align: center;">동기 (SqliteSaver)</th>
<th style="text-align: center;">비동기 (AsyncSqliteSaver)</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">CLI 스크립트</td>
<td style="text-align: center;">✅ 권장</td>
<td style="text-align: center;">❌ 불필요</td>
</tr>
<tr>
<td style="text-align: left;">FastAPI 웹 서버</td>
<td style="text-align: center;">❌ 블로킹</td>
<td style="text-align: center;">✅ 필수</td>
</tr>
<tr>
<td style="text-align: left;">동시 사용자 처리</td>
<td style="text-align: center;">❌ 느림</td>
<td style="text-align: center;">✅ 빠름</td>
</tr>
<tr>
<td style="text-align: left;">간단한 테스트</td>
<td style="text-align: center;">✅ 편함</td>
<td style="text-align: center;">❌ 복잡함</td>
</tr>
</tbody>
</table>

<br>

In [15]:
# pip install langgraph-checkpoint-sqlite aiosqlite==0.21.0

In [21]:
import asyncio
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.agents import create_agent

- 비동기 그래프 정의

In [22]:
async def chatbot(state: MessagesState):
    """비동기 챗봇 노드"""
    agent = create_agent(model="openai:gpt-4.1-mini", tools=[])
    # ainvoke로 비동기 호출
    response = await agent.ainvoke(state)
    return {"messages": response["messages"]}

In [23]:
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

<br>

- **비동기 `main()` 함수**

In [27]:
async def main():
    # async with로 비동기 context manager 사용
    async with AsyncSqliteSaver.from_conn_string("checkpoints.db") as checkpointer:
        # 그래프 컴파일
        graph = builder.compile(checkpointer=checkpointer)

        # 비동기 그래프 실행
        config = {"configurable": {"thread_id": "user-001"}}

        # ainvoke로 비동기 호출
        response = await graph.ainvoke(
            {"messages": [{"role": "user", "content": "안녕하세요"}]},
            config=config
        )
        print(response["messages"][-1].content)

        # 비동기 상태 조회
        # aget_state로 비동기 상태 조회
        state = await graph.aget_state(config)
        print(f"메시지 수: {len(state.values['messages'])}")

- 실행

In [28]:
# asyncio.run(main())

await main()

안녕하세요! 무엇을 도와드릴까요?
메시지 수: 2


<br>

#### 여러 사용자 동시 처리
- **동기 vs 비동기 처리 시간**:
    - 동기 (순차 처리): 사용자 10명 × 2초 = 20초
    - 비동기 (병렬 처리): 약 2~3초 (동시 처리)
    
    $\rightarrow$ **웹 서버에서 동시 사용자를 처리해야 한다면 비동기가 필수**

In [29]:
import asyncio
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.agents import create_agent

- 그래프 구성

In [30]:
async def chatbot(state: MessagesState):
    """비동기 챗봇 노드"""
    agent = create_agent(model="openai:gpt-4.1-mini", tools=[])
    response = await agent.ainvoke(state)
    return {"messages": response["messages"]}

In [31]:
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

- 단일 사용자 처리 함수

In [32]:
async def process_user(graph, user_id: str, message: str):
    """단일 사용자의 메시지를 처리"""
    config = {"configurable": {"thread_id": user_id}}

    response = await graph.ainvoke(
        {"messages": [{"role": "user", "content": message}]},
        config=config
    )

    print(f"[{user_id}] {response['messages'][-1].content}")

- 여러 사용자 동시 처리

In [36]:
async def main():
    async with AsyncSqliteSaver.from_conn_string("./DB/checkpoints.db") as checkpointer:
        graph = builder.compile(checkpointer=checkpointer)

        # asyncio.gather로 10명 동시 처리
        # 순차 처리 대비 약 10배 빠름
        tasks = [
            process_user(graph, f"user-{i:03d}", f"안녕하세요, 저는 사용자 {i}입니다")
            for i in range(1, 11)
        ]

        # 모든 작업이 완료될 때까지 대기
        await asyncio.gather(*tasks)

In [37]:
# asyncio.run(main())

await main()

[user-001] 안녕하세요, 사용자 1님! 만나서 반갑습니다. 어떻게 도와드릴까요?
[user-010] 안녕하세요, 사용자 10님! 무엇을 도와드릴까요?
[user-002] 안녕하세요, 사용자 2님! 만나서 반갑습니다. 무엇을 도와드릴까요?
[user-003] 안녕하세요, 사용자 3님! 어떻게 도와드릴까요?
[user-009] 안녕하세요, 사용자 9님! 어떻게 도와드릴까요?
[user-006] 안녕하세요, 사용자 6님! 어떻게 도와드릴까요?
[user-004] 안녕하세요, 사용자 4님! 어떻게 도와드릴까요?
[user-005] 안녕하세요, 사용자 5님! 어떻게 도와드릴까요?
[user-007] 안녕하세요, 사용자 7님! 어떻게 도와드릴까요?
[user-008] 안녕하세요, 사용자 8님! 어떻게 도와드릴까요?


<br>

<hr>

<br>

### Time Travel - 과거 상태로 이동
- **Time Travel은 LangGraph에서 그래프 실행의 과거 체크포인트로 돌아가 다른 경로를 탐색할 수 있게 해주는 강력한 디버깅 및 탐색 기능**

<img src='img/1-10-1.png' width=800>

<br>

#### Time Travel의 필요성
- **AI 에이전트는 종종 비결정적(Non-deterministic) 특성을 가지므로, 동일한 입력에 대해 다른 결과를 생성할 수 있음**

<br>

- **추론 과정 이해 (Understand Reasoning)**
  - 성공적인 결과로 이끈 의사결정 단계를 역추적하여 분석
  - 각 단계에서 어떤 선택이 이루어졌는지 검토
  - 모델의 추론 패턴과 전략 파악

- **오류 디버깅 (Debug Mistakes)**
  - 잘못된 결과가 발생한 정확한 지점 식별
  - 오류 원인을 단계별로 추적하여 근본 원인 분석
  - 문제가 발생한 특정 노드와 상태 변경 확인

- **대안 탐색 (Explore Alternatives)**
  - 동일한 출발점에서 다른 경로를 시도하여 더 나은 솔루션 발견
  - A/B 테스팅을 통한 최적의 전략 선택
  - 여러 가능성을 탐색하여 모델 성능 개선

<br>

#### Time Travel의 핵심 원리: Fork (분기)
- **과거 체크포인트에서 실행을 제개하면, 원래 실행 이력은 그대로 보존되고, 새로운 분기 (Fork)가 생성**

<br>

- 원래 이력(`CP0` $\rightarrow$ `CP1` $\rightarrow$ `CP2` $\rightarrow$ `CP3`)은 삭제되지 않고 보존
- CP1에서 분기하여 새 이력(`CP4` $\rightarrow$ `CP5`)이 생성
  
  $\rightarrow$ 이 덕분에 원래 결과와 새 결과를 비교 분석 가능


<img src='./img/1-10-2.png' width=300>

<br>

#### Time Travel의 구현

<br>

**Step 1: 체크포인터와 함께 그래프 실행**

<br>

**Step 2: 체크포인트 히스토리 조회**

<br>

**Step 3: 상태 수정 (선택적)**

<br>

**Step 4: 체크포인트에서 실행 재개**

<br>
